In [ ]:
#!/usr/bin/env python3
"""
Colocalization Analysis (coloc.abf)
=====================================
Python merges GWAS + eQTL data per region, then calls R coloc.abf.
Applicable to any GWAS vs SingleBrain eQTL colocalization.
"""

import os, time, subprocess, tempfile
import numpy as np, pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
from io import StringIO


# ========================= Configuration =========================

# --- GWAS data ---
GWAS_FILE     = "./data/gwas/your_gwas_summary_stats.txt"          # Path to GWAS summary statistics
GWAS_NAME     = "demo_trait"          # Short trait name (used in output filenames)
GWAS_SEP      = "\t"
GWAS_COL_SNP  = "SNP"
GWAS_COL_CHR  = "chr"
GWAS_COL_BP   = "pos"
GWAS_COL_BETA = "beta"
GWAS_COL_SE   = "se"
GWAS_COL_PVAL = "pval"
GWAS_COL_EA   = "effect_allele"
GWAS_COL_EAF  = "eaf"
GWAS_COL_N    = "samplesize"
GWAS_TYPE     = "quant"     # "quant" or "cc"

# --- MR strict results (gene-cell pairs to colocalize) ---
STRICT_FILE = "./data/mr/strict_for_coloc.csv"            # Path to strict_for_coloc.csv from csMR

# --- SingleBrain eQTL ---
SINGLEBRAIN_DIR = "./data/visium/"        # Directory containing {celltype}_eqtl_full_assoc.tsv.gz

# --- GTF annotation ---
GTF_FILE = "./data/annotation/gencode.vsj.annotation.gtf"               # Path to gencode GTF file

# --- Parameters ---
WINDOW      = 500000        # Region: TSS +/- this many bp
MIN_SNPS    = 50
N_EQTL      = 983
NUM_WORKERS = 36
OUTPUT_DIR  = "./results/"            # Output directory


# ========================= R coloc Script =========================

R_COLOC_SCRIPT = """
suppressMessages(library(coloc))
args <- commandArgs(trailingOnly=TRUE)
merged_file <- args[1]
output_file <- args[2]
gtype <- args[3]
n_gwas <- as.numeric(args[4])
n_eqtl <- as.numeric(args[5])

m <- read.csv(merged_file, stringsAsFactors=FALSE)

if (nrow(m) < 50) {
    write.csv(data.frame(n_snps=nrow(m), PP_H0=NA, PP_H1=NA, PP_H2=NA,
                         PP_H3=NA, PP_H4=NA, best_snp=NA, best_snp_pp=NA),
              output_file, row.names=FALSE)
    quit(save="no")
}

tryCatch({
    if (gtype == "cc") {
        d1 <- list(beta=m$BETA_gwas, varbeta=m$SE_gwas^2,
                   snp=m$SNP, position=m$POS, type="cc", N=n_gwas)
    } else {
        d1 <- list(beta=m$BETA_gwas, varbeta=m$SE_gwas^2,
                   snp=m$SNP, position=m$POS, type="quant", N=n_gwas, sdY=1)
    }
    if ("MAF_gwas" %in% names(m) && !any(is.na(m$MAF_gwas))) {
        d1$MAF <- m$MAF_gwas
    }

    d2 <- list(beta=m$BETA_eqtl, varbeta=m$SE_eqtl^2,
               snp=m$SNP, position=m$POS, type="quant", N=n_eqtl, sdY=1)

    res <- coloc.abf(d1, d2, p1=1e-4, p2=1e-4, p12=1e-5)

    pp <- res$results
    bi <- which.max(pp$SNP.PP.H4)

    write.csv(data.frame(
        n_snps = nrow(m),
        PP_H0 = res$summary["PP.H0.abf"],
        PP_H1 = res$summary["PP.H1.abf"],
        PP_H2 = res$summary["PP.H2.abf"],
        PP_H3 = res$summary["PP.H3.abf"],
        PP_H4 = res$summary["PP.H4.abf"],
        best_snp = pp$snp[bi],
        best_snp_pp = pp$SNP.PP.H4[bi]
    ), output_file, row.names=FALSE)

}, error=function(e) {
    write.csv(data.frame(n_snps=nrow(m), PP_H0=NA, PP_H1=NA, PP_H2=NA,
                         PP_H3=NA, PP_H4=NA, best_snp=NA, best_snp_pp=NA),
              output_file, row.names=FALSE)
})
"""


# ========================= Utilities =========================

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")


# ========================= Per-Locus Colocalization =========================

def coloc_one(args):
    """Run colocalization for one gene x cell type pair."""
    gene, ensg, ct, gene_chr, gene_tss, full_assoc, gwas_chr_data, n_gwas, r_script, has_eaf = args

    try:
        region_start = max(0, gene_tss - WINDOW)
        region_end = gene_tss + WINDOW
        chr_key = str(gene_chr)

        # --- Extract eQTL in region ---
        cmd = f"zgrep '{ensg}' {full_assoc}"
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=120)
        if not result.stdout.strip():
            return None

        header = subprocess.run(f"zcat {full_assoc} | head -1",
                                shell=True, capture_output=True, text=True).stdout.strip()
        eqtl = pd.read_csv(StringIO(header + "\n" + result.stdout), sep="\t")
        eqtl["pos_int"] = eqtl["pos"].astype(int)
        eqtl = eqtl[(eqtl["pos_int"] >= region_start) & (eqtl["pos_int"] <= region_end)]
        eqtl = eqtl[(eqtl["fixed_beta"] != 0) & (eqtl["fixed_sd"] > 0)]
        eqtl = eqtl.drop_duplicates(subset="variant_id", keep="first")

        if len(eqtl) < MIN_SNPS:
            return None

        # --- Extract GWAS in region ---
        if chr_key not in gwas_chr_data:
            return None
        gwas = gwas_chr_data[chr_key]
        gwas_region = gwas[(gwas[GWAS_COL_BP] >= region_start) & (gwas[GWAS_COL_BP] <= region_end)]
        gwas_region = gwas_region[gwas_region[GWAS_COL_SE] > 0]
        gwas_region = gwas_region.drop_duplicates(subset=GWAS_COL_SNP, keep="first")

        if len(gwas_region) < MIN_SNPS:
            return None

        # --- Merge in Python ---
        eqtl_df = eqtl[["variant_id", "pos_int", "fixed_beta", "fixed_sd"]].copy()
        eqtl_df.columns = ["SNP", "POS", "BETA_eqtl", "SE_eqtl"]

        gwas_cols = [GWAS_COL_SNP, GWAS_COL_BP, GWAS_COL_BETA, GWAS_COL_SE]
        gwas_df = gwas_region[gwas_cols].copy()
        gwas_df.columns = ["SNP", "POS_gwas", "BETA_gwas", "SE_gwas"]

        if has_eaf:
            gwas_df["MAF_gwas"] = gwas_region[GWAS_COL_EAF].values
            gwas_df.loc[gwas_df["MAF_gwas"] > 0.5, "MAF_gwas"] = \
                1 - gwas_df.loc[gwas_df["MAF_gwas"] > 0.5, "MAF_gwas"]

        merged = eqtl_df.merge(gwas_df, on="SNP")
        merged = merged.drop_duplicates(subset="SNP", keep="first")
        merged = merged.dropna()
        merged = merged[(merged["SE_gwas"] > 0) & (merged["SE_eqtl"] > 0)]

        if len(merged) < MIN_SNPS:
            return None

        # --- Write merged data and call R ---
        with tempfile.TemporaryDirectory() as tmp:
            merged_file = os.path.join(tmp, "merged.csv")
            merged.to_csv(merged_file, index=False)

            result_file = os.path.join(tmp, "result.csv")
            cmd = f"Rscript {r_script} {merged_file} {result_file} {GWAS_TYPE} {n_gwas} {N_EQTL}"
            subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=300)

            if not os.path.exists(result_file):
                return None
            res = pd.read_csv(result_file)
            if len(res) == 0 or pd.isna(res["PP_H4"].values[0]):
                return None

            return {
                "gene": gene, "ensg": ensg, "cell_type": ct,
                "region": f"chr{gene_chr}:{region_start}-{region_end}",
                "n_snps": int(res["n_snps"].values[0]),
                "PP_H0": round(res["PP_H0"].values[0], 6),
                "PP_H1": round(res["PP_H1"].values[0], 6),
                "PP_H2": round(res["PP_H2"].values[0], 6),
                "PP_H3": round(res["PP_H3"].values[0], 6),
                "PP_H4": round(res["PP_H4"].values[0], 6),
                "PP3_PP4": round(res["PP_H3"].values[0] + res["PP_H4"].values[0], 6),
                "PP4_ratio": round(res["PP_H4"].values[0] /
                                   max(res["PP_H3"].values[0] + res["PP_H4"].values[0], 1e-10), 6),
                "best_snp": res["best_snp"].values[0],
                "best_snp_pp": round(res["best_snp_pp"].values[0], 6)
                    if pd.notna(res["best_snp_pp"].values[0]) else 0,
            }
    except Exception:
        return None


# ========================= Main =========================

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    log("=" * 70)
    log(f"Colocalization Analysis (coloc.abf)")
    log(f"GWAS: {GWAS_NAME} ({GWAS_TYPE}), region: TSS +/-{WINDOW // 1000}kb")
    log("=" * 70)

    # Write R script to disk
    r_script = os.path.join(OUTPUT_DIR, "run_coloc.R")
    with open(r_script, "w") as f:
        f.write(R_COLOC_SCRIPT)

    tr = subprocess.run(
        'Rscript -e "library(coloc); cat(as.character(packageVersion(\'coloc\')),\'\\n\')"',
        shell=True, capture_output=True, text=True)
    log(f"R coloc version: {tr.stdout.strip()}")

    # --- [1] Gene coordinates ---
    log("\n[1/4] Gene coordinates...")
    cache_gene = os.path.join(OUTPUT_DIR, "gene_coords.csv")
    if os.path.exists(cache_gene):
        gene_df = pd.read_csv(cache_gene)
    else:
        cmd = f"grep -P '\\tgene\\t' {GTF_FILE}"
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        recs = []
        for l in r.stdout.strip().split("\n"):
            p = l.split("\t")
            if len(p) < 9:
                continue
            chrom = p[0].replace("chr", ""); start = int(p[3]); end = int(p[4]); strand = p[6]
            gid = gn = ""
            for fi in p[8].split(";"):
                fi = fi.strip()
                if fi.startswith("gene_id"):
                    gid = fi.split('"')[1] if '"' in fi else ""
                elif fi.startswith("gene_name"):
                    gn = fi.split('"')[1] if '"' in fi else ""
            if gid and gn:
                tss = start if strand == "+" else end
                recs.append({"ensg": gid, "ensg_base": gid.split(".")[0],
                             "symbol": gn, "chr": chrom, "tss": tss})
        gene_df = pd.DataFrame(recs).drop_duplicates(subset="symbol", keep="first")
        gene_df.to_csv(cache_gene, index=False)
    log(f"  {len(gene_df)} genes")

    # --- [2] MR strict results ---
    log("\n[2/4] Loading MR strict results...")
    strict = pd.read_csv(STRICT_FILE)
    log(f"  {len(strict)} pairs, {strict['gene'].nunique()} genes")

    gene_coords = dict(zip(gene_df["symbol"],
                           zip(gene_df["chr"], gene_df["tss"], gene_df["ensg_base"])))

    # --- [3] Load GWAS ---
    log("\n[3/4] Loading GWAS...")
    gwas = pd.read_csv(GWAS_FILE, sep=GWAS_SEP)
    n_gwas = gwas[GWAS_COL_N].median()
    has_eaf = GWAS_COL_EAF in gwas.columns

    gwas_by_chr = {}
    for chrom in gwas[GWAS_COL_CHR].unique():
        gwas_by_chr[str(chrom)] = gwas[gwas[GWAS_COL_CHR] == chrom].copy()
    log(f"  {len(gwas)} SNPs, N={n_gwas:.0f}, MAF={'yes' if has_eaf else 'no'}")

    # --- [4] Run colocalization ---
    log("\n[4/4] Running colocalization...")
    tasks = []
    for _, row in strict.iterrows():
        gene = row["gene"]; ct = row["cell_type"]
        if gene not in gene_coords:
            continue
        gene_chr, gene_tss, ensg_base = gene_coords[gene]
        full_assoc = os.path.join(SINGLEBRAIN_DIR, f"{ct}_eqtl_full_assoc.tsv.gz")
        if not os.path.exists(full_assoc):
            continue
        tasks.append((gene, ensg_base, ct, gene_chr, gene_tss,
                      full_assoc, gwas_by_chr, n_gwas, r_script, has_eaf))

    log(f"  Tasks: {len(tasks)}")

    results = []; done = 0; t0 = time.time()
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as exe:
        futs = {exe.submit(coloc_one, t): t for t in tasks}
        for fut in as_completed(futs):
            r = fut.result(); done += 1
            if r:
                results.append(r)
                if r["PP_H4"] > 0.5:
                    log(f"  * {r['gene']:15s} in {r['cell_type']:10s} "
                        f"PP4={r['PP_H4']:.3f} PP3={r['PP_H3']:.3f} "
                        f"n={r['n_snps']} SNP={r['best_snp']}")
            if done % 100 == 0:
                log(f"  Progress: {done}/{len(tasks)} "
                    f"({(time.time() - t0) / 60:.1f} min, valid: {len(results)})")

    elapsed = (time.time() - t0) / 60
    log(f"\nCompleted: {elapsed:.1f} min, valid: {len(results)}")

    if not results:
        log("No valid results")
        return

    df = pd.DataFrame(results).sort_values("PP_H4", ascending=False)

    # Summary statistics
    log(f"\n{'=' * 70}")
    log(f"Colocalization results: {GWAS_NAME} vs eQTL")
    log(f"{'=' * 70}")
    log(f"Total tests: {len(df)}")
    log(f"PP4 > 0.8 (strict):    {(df['PP_H4'] > 0.8).sum()}")
    log(f"PP4 > 0.7:             {(df['PP_H4'] > 0.7).sum()}")
    log(f"PP4 > 0.5:             {(df['PP_H4'] > 0.5).sum()}")
    log(f"PP3+PP4>0.8 & PP4>PP3: {((df['PP3_PP4'] > 0.8) & (df['PP4_ratio'] > 0.5)).sum()}")

    log(f"\nPP4 distribution: mean={df['PP_H4'].mean():.4f}, "
        f"median={df['PP_H4'].median():.4f}, max={df['PP_H4'].max():.4f}")

    # Dominant hypothesis
    df["max_hypothesis"] = df[["PP_H0", "PP_H1", "PP_H2", "PP_H3", "PP_H4"]].idxmax(axis=1)
    log(f"\nDominant hypothesis:\n{df['max_hypothesis'].value_counts().to_string()}")

    # Top results
    log(f"\nTop 30 (by PP4):")
    for _, r in df.head(30).iterrows():
        log(f"  {r['gene']:15s} in {r['cell_type']:10s} "
            f"PP4={r['PP_H4']:.4f} PP3={r['PP_H3']:.4f} PP1={r['PP_H1']:.4f} "
            f"n={r['n_snps']} SNP={r['best_snp']}")

    # Merge with MR results
    strict_mr = pd.read_csv(STRICT_FILE)
    merged = df.merge(strict_mr[["gene", "cell_type", "ivw_beta", "ivw_fdr", "direction"]],
                      on=["gene", "cell_type"], how="left")

    both = merged[merged["PP_H4"] > 0.5]
    if len(both) > 0:
        log(f"\nMR significant + PP4 > 0.5:")
        for _, r in both.iterrows():
            log(f"  {r['gene']:15s} in {r['cell_type']:10s} "
                f"{r.get('direction', '?')} MR_FDR={r.get('ivw_fdr', 1):.2e} "
                f"PP4={r['PP_H4']:.3f} PP3={r['PP_H3']:.3f}")

    # Save results
    df.to_csv(os.path.join(OUTPUT_DIR, f"coloc_{GWAS_NAME}_all.csv"), index=False)
    merged.to_csv(os.path.join(OUTPUT_DIR, f"coloc_{GWAS_NAME}_merged_mr.csv"), index=False)
    sig = df[df["PP_H4"] > 0.5]
    if len(sig) > 0:
        sig.to_csv(os.path.join(OUTPUT_DIR, f"coloc_{GWAS_NAME}_pp4_gt05.csv"), index=False)

    log(f"\nAll results saved to: {OUTPUT_DIR}")
    log("Colocalization completed.")


if __name__ == "__main__":
    main()